# Week 1 — First naive line simulation

## Goal

Build the smallest possible working model of local cross-correction.

The idea is simple:

- some nuclei are corrected
- corrected nuclei produce signal
- signal spreads to nearby nuclei
- if enough signal accumulates locally, rescue may happen

This is a toy model, not yet a biologically realistic simulator.

## Why a 1D line?

A multinucleated muscle fibre is spatial. The easiest spatial abstraction is a line of nuclei.

That means:
- one list index = one nucleus
- `i - 1` is the left neighbour
- `i + 1` is the right neighbour

I chose this because it is easy to inspect and debug.

In [ ]:
from __future__ import annotations

def run_naive_line(n_nodes=20, corrected_fraction=0.1, steps=50):
    signal = [0.0 for _ in range(n_nodes)]
    corrected = [False for _ in range(n_nodes)]
    n_corrected = max(1, round(n_nodes * corrected_fraction))
    for i in range(n_corrected):
        corrected[i] = True

    rescued = [False for _ in range(n_nodes)]

    for _ in range(steps):
        next_signal = signal[:]
        for i in range(n_nodes):
            left = signal[i - 1] if i > 0 else signal[i]
            right = signal[i + 1] if i < n_nodes - 1 else signal[i]
            production = 1.0 if corrected[i] else 0.0
            next_signal[i] = 0.6 * signal[i] + 0.2 * left + 0.2 * right + production * 0.05
            if next_signal[i] > 1.0:
                rescued[i] = True
        signal = next_signal

    return {"signal": signal, "rescued": rescued}

## Intuition behind the update rule

The line

`0.6 * current + 0.2 * left + 0.2 * right + production * 0.05`

means:

- keep some current signal
- mix in neighbour signal
- inject a little fresh signal at corrected nuclei

This is not yet derived from a diffusion equation. It is a first intuition-building rule.

In [ ]:
result = run_naive_line()
result

## What I learned

- A biological hypothesis can be translated into a small iterative system.
- Local coupling is enough to test whether rescue amplification is plausible.
- The coefficients are heuristic, so the next step should be a more principled transport model.